In [ ]:
!ls /content

LBP.zip  sample_data


In [ ]:
!rm -rf /content/LBP
!unzip -q /content/LBP.zip -d /content

%cd /content/LBP
!ls

/content/LBP
experiments  Makefile	 requirements-dev.txt  tests
lbpface      pytest.ini  requirements.txt


In [ ]:
!pip install -q pytest scikit-image Pillow

In [ ]:
import numpy as np
import scipy
import PIL

print("numpy:", np.__version__)
print("scipy:", scipy.__version__)
print("Pillow:", PIL.__version__)

numpy: 2.1.3
scipy: 1.16.3
Pillow: 11.3.0


In [ ]:
!rm -rf /content/orl_src

!git clone --depth 1 \
    https://github.com/saeid436/Face-Recognition-MLP.git \
    /content/orl_src

!mkdir -p /content/LBP/data
!cp -r /content/orl_src/ORL /content/LBP/data/ORL

Cloning into '/content/orl_src'...
remote: Enumerating objects: 453, done.
remote: Counting objects: 100% (453/453), done.
remote: Compressing objects: 100% (452/452), done.
remote: Total 453 (delta 1), reused 453 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (453/453), 3.57 MiB | 7.91 MiB/s, done.
Resolving deltas: 100% (1/1), done.


In [ ]:
!ls data/ORL | head

FeatureExtraction.m
Maxlikelihood2.m
Maxlikelihood.m
Randomize.m
README
s1
s10
s11
s12
s13


In [ ]:
!python -m lbpface.datasets data/ORL

images      : 400 of size 92x112
subjects    : 40
fingerprint : 9972801133ec8ba8b06221a0f2c7c087c2e2a8d99a5fdd0ddfbf90beb82d5ac3


In [ ]:
!pytest -q -k "not bit_exact_reproduction_of_the_stored_run"

........................................................................ [ 82%]
...............                                                          [100%]
87 passed, 1 deselected in 25.60s


In [ ]:
!python -m experiments.orl \
    --data data/ORL \
    --seed 0 \
    --out results/orl_seed0.json

ORL: 400 images, feature length 2187, features+distances 6.9s
mean recognition rate 0.9826  std 0.0096  95% [0.962, 1.000]  (100 permutations)
paper reports         0.9800  std 0.0120
wrote results/orl_seed0.json   digest 3d349dac87dc50db...


In [ ]:
import numpy as np
import pandas as pd

from scipy import ndimage

from lbpface.datasets import load_orl
from lbpface.features import LBPDescriptor
from lbpface.distances import pairwise_distances
from lbpface.classify import recognition_rate


# ============================================================
# 1. Load ORL
# ============================================================

fs = load_orl("data/ORL")

print("Images:", fs.images.shape)
print("Subjects:", len(np.unique(fs.subjects)))


# ============================================================
# 2. Baseline descriptor
#
# LBP^u2_(16,2)
# window = 30 x 37
# 92x112 image -> 3x3 regions
# ============================================================

desc = LBPDescriptor.orl()


# ============================================================
# 3. EXACT fixed split from the provided code
#
# image 1-5  -> gallery
# image 6-10 -> probe
# ============================================================

is_gallery = (np.arange(len(fs.images)) % 10) < 5

gal = fs.images[is_gallery]
prb = fs.images[~is_gallery]

gl = fs.subjects[is_gallery]
pl = fs.subjects[~is_gallery]

print("Gallery:", gal.shape)
print("Probe:", prb.shape)


# ============================================================
# 4. Compute gallery features ONCE
# ============================================================

G = desc.transform(gal)

print("Gallery feature matrix:", G.shape)
print("Feature length:", G.shape[1])


# ============================================================
# 5. Scoring function
# ============================================================

def score(probe_images):

    P = desc.transform(probe_images)

    distances = pairwise_distances(
        P,
        G,
        "chi2"
    )

    return recognition_rate(
        distances,
        gl,
        pl
    )


# ============================================================
# 6. Exact illumination transformation
# ============================================================

def illum_ramp(img, s):

    ramp = 1 + s * np.linspace(
        -1,
        1,
        img.shape[1]
    )

    return np.clip(
        img * ramp,
        0,
        255
    )


# ============================================================
# 7. Exact rotation transformation
# ============================================================

def rotate(img, deg):

    return ndimage.rotate(
        img.astype(float),
        deg,
        reshape=False,
        order=1,
        mode="nearest"
    )

Images: (400, 112, 92)
Subjects: 40
Gallery: (200, 112, 92)
Probe: (200, 112, 92)
Gallery feature matrix: (200, 2187)
Feature length: 2187


In [ ]:
results = []

# Normal
results.append(
    ("Normal", score(prb))
)

# Uniform brightness
results.append(
    (
        "Uniform brightness",
        score([0.5 * i + 20 for i in prb])
    )
)

# Lighting gradient
for s in [0.2, 0.4, 0.6, 0.8]:

    acc = score([
        illum_ramp(i, s)
        for i in prb
    ])

    results.append(
        (f"Lighting ramp {s}", acc)
    )


# Rotation
for d in [5, 10, 20]:

    acc = score([
        rotate(i, d)
        for i in prb
    ])

    results.append(
        (f"Rotation {d}°", acc)
    )


baseline_df = pd.DataFrame(
    results,
    columns=["Condition", "Accuracy"]
)

baseline_df

,Condition,Accuracy
0,Normal,0.975
1,Uniform brightness,0.975
2,Lighting ramp 0.2,0.960
3,Lighting ramp 0.4,0.945
4,Lighting ramp 0.6,0.895
5,Lighting ramp 0.8,0.705
6,Rotation 5°,0.970
7,Rotation 10°,0.925
8,Rotation 20°,0.630


Trying Improvement

In [ ]:
# ============================================================
# Improvement 1:
# Rotation-compensated matching
# ============================================================

corrective_angles = [
    -20, -15, -10, -5,
      0,
      5,  10, 15, 20
]


def score_rotation_compensated(probe_images):

    all_distances = []

    for angle in corrective_angles:

        # Rotate every probe by a possible corrective angle
        corrected = [
            rotate(img, angle)
            if angle != 0
            else img
            for img in probe_images
        ]

        # Compute LBP features
        P = desc.transform(corrected)

        # Distance to ORIGINAL gallery
        D = pairwise_distances(
            P,
            G,
            "chi2"
        )

        all_distances.append(D)

    # Shape:
    # angles × probes × gallery
    all_distances = np.stack(
        all_distances,
        axis=0
    )

    # For every probe-gallery pair,
    # choose the best angle
    best_distances = np.min(
        all_distances,
        axis=0
    )

    return recognition_rate(
        best_distances,
        gl,
        pl
    )

In [ ]:
normal_rotcomp = score_rotation_compensated(prb)

print("Baseline Normal:", 0.975)
print("Rotation-compensated Normal:", normal_rotcomp)

Baseline Normal: 0.975
Rotation-compensated Normal: 0.975


In [ ]:
rotation_results = []

for d in [5, 10, 20]:

    rotated_probes = [
        rotate(img, d)
        for img in prb
    ]

    baseline_acc = score(
        rotated_probes
    )

    improved_acc = score_rotation_compensated(
        rotated_probes
    )

    rotation_results.append({
        "Rotation": f"{d}°",
        "Baseline": baseline_acc,
        "Improved": improved_acc,
        "Gain": improved_acc - baseline_acc
    })


rotation_df = pd.DataFrame(rotation_results)

rotation_df

,Rotation,Baseline,Improved,Gain
0,5°,0.970,0.97,0.000
1,10°,0.925,0.94,0.015
2,20°,0.630,0.87,0.240


In [ ]:
# ============================================================
# Improvement 2:
# Local illumination normalization / Single-Scale Retinex
# ============================================================

def illumination_normalize(img, sigma=8):

    img = np.asarray(img, dtype=np.float32)

    # log transform:
    # multiplicative illumination becomes approximately additive
    log_img = np.log1p(img)

    # estimate slowly-varying illumination field
    illumination = ndimage.gaussian_filter(
        log_img,
        sigma=sigma
    )

    # remove illumination
    normalized = log_img - illumination

    # rescale to [0, 255]
    # percentile clipping avoids a few extreme pixels
    lo, hi = np.percentile(
        normalized,
        [1, 99]
    )

    normalized = (
        normalized - lo
    ) / (
        hi - lo + 1e-8
    )

    normalized = np.clip(
        normalized,
        0,
        1
    )

    return normalized * 255.0

In [ ]:
# Normalize gallery images once

sigma = 8

gal_norm = [
    illumination_normalize(img, sigma=sigma)
    for img in gal
]

G_norm = desc.transform(gal_norm)

print("Normalized gallery feature matrix:", G_norm.shape)

Normalized gallery feature matrix: (200, 2187)


In [ ]:
def score_illumination_normalized(
    probe_images,
    sigma=8
):

    # normalize probe images
    probe_norm = [
        illumination_normalize(
            img,
            sigma=sigma
        )
        for img in probe_images
    ]

    # LBP features
    P_norm = desc.transform(
        probe_norm
    )

    # compare normalized probe
    # against normalized gallery
    D = pairwise_distances(
        P_norm,
        G_norm,
        "chi2"
    )

    return recognition_rate(
        D,
        gl,
        pl
    )

In [ ]:
normal_illum = score_illumination_normalized(
    prb
)

print("Baseline Normal:", 0.975)
print(
    "Illumination-normalized Normal:",
    normal_illum
)

Baseline Normal: 0.975
Illumination-normalized Normal: 0.97


In [ ]:
lighting_results = []

# ------------------------------------------------------------
# Normal
# ------------------------------------------------------------

baseline_acc = score(prb)

improved_acc = score_illumination_normalized(
    prb
)

lighting_results.append({
    "Condition": "Normal",
    "Baseline": baseline_acc,
    "Improved": improved_acc,
    "Gain": improved_acc - baseline_acc
})


# ------------------------------------------------------------
# Uniform brightness
# ------------------------------------------------------------

uniform_probe = [
    0.5 * img + 20
    for img in prb
]

baseline_acc = score(
    uniform_probe
)

improved_acc = score_illumination_normalized(
    uniform_probe
)

lighting_results.append({
    "Condition": "Uniform brightness",
    "Baseline": baseline_acc,
    "Improved": improved_acc,
    "Gain": improved_acc - baseline_acc
})


# ------------------------------------------------------------
# Lighting ramps
# ------------------------------------------------------------

for s in [0.2, 0.4, 0.6, 0.8]:

    corrupted = [
        illum_ramp(img, s)
        for img in prb
    ]

    baseline_acc = score(
        corrupted
    )

    improved_acc = score_illumination_normalized(
        corrupted
    )

    lighting_results.append({
        "Condition": f"Lighting ramp {s}",
        "Baseline": baseline_acc,
        "Improved": improved_acc,
        "Gain": improved_acc - baseline_acc
    })


lighting_df = pd.DataFrame(
    lighting_results
)

lighting_df

,Condition,Baseline,Improved,Gain
0,Normal,0.975,0.970,-0.005
1,Uniform brightness,0.975,0.975,0.000
2,Lighting ramp 0.2,0.960,0.965,0.005
3,Lighting ramp 0.4,0.945,0.970,0.025
4,Lighting ramp 0.6,0.895,0.970,0.075
5,Lighting ramp 0.8,0.705,0.970,0.265


In [ ]:
def score_illumination_fused(
    probe_images,
    alpha=0.5,
    sigma=8
):
    # --------------------------------------------------------
    # Raw LBP
    # --------------------------------------------------------
    P_raw = desc.transform(
        probe_images
    )

    D_raw = pairwise_distances(
        P_raw,
        G,
        "chi2"
    )

    # --------------------------------------------------------
    # Illumination-normalized LBP
    # --------------------------------------------------------
    probe_norm = [
        illumination_normalize(
            img,
            sigma=sigma
        )
        for img in probe_images
    ]

    P_norm = desc.transform(
        probe_norm
    )

    D_norm = pairwise_distances(
        P_norm,
        G_norm,
        "chi2"
    )

    # --------------------------------------------------------
    # Distance fusion
    # --------------------------------------------------------
    D_fused = (
        alpha * D_raw
        +
        (1 - alpha) * D_norm
    )

    return recognition_rate(
        D_fused,
        gl,
        pl
    )

In [ ]:
normal_fused = score_illumination_fused(
    prb,
    alpha=0.5
)

print("Baseline Normal:      ", 0.975)
print("Normalized only:      ", 0.970)
print("Raw + normalized:     ", normal_fused)

Baseline Normal:       0.975
Normalized only:       0.97
Raw + normalized:      0.975


In [ ]:
fusion_results = []

# Normal
fusion_results.append({
    "Condition": "Normal",
    "Baseline": score(prb),
    "Normalized": score_illumination_normalized(prb),
    "Fused": score_illumination_fused(prb)
})


# Uniform brightness
uniform_probe = [
    0.5 * img + 20
    for img in prb
]

fusion_results.append({
    "Condition": "Uniform brightness",
    "Baseline": score(uniform_probe),
    "Normalized": score_illumination_normalized(uniform_probe),
    "Fused": score_illumination_fused(uniform_probe)
})


# Lighting ramps
for s in [0.2, 0.4, 0.6, 0.8]:

    corrupted = [
        illum_ramp(img, s)
        for img in prb
    ]

    fusion_results.append({
        "Condition": f"Lighting ramp {s}",
        "Baseline": score(corrupted),
        "Normalized": score_illumination_normalized(corrupted),
        "Fused": score_illumination_fused(corrupted)
    })


fusion_df = pd.DataFrame(
    fusion_results
)

fusion_df

,Condition,Baseline,Normalized,Fused
0,Normal,0.975,0.970,0.975
1,Uniform brightness,0.975,0.975,0.975
2,Lighting ramp 0.2,0.960,0.965,0.970
3,Lighting ramp 0.4,0.945,0.970,0.965
4,Lighting ramp 0.6,0.895,0.970,0.975
5,Lighting ramp 0.8,0.705,0.970,0.965


In [ ]:
# ============================================================
# Final Robust LBP
#
# 1. Multi-angle rotation compensation
# 2. Raw + illumination-normalized distance fusion
# ============================================================

corrective_angles = [
    -20, -15, -10, -5,
      0,
      5,  10, 15, 20
]


def score_final_robust(
    probe_images,
    alpha=0.5,
    sigma=8
):

    all_fused_distances = []

    # --------------------------------------------------------
    # Try every possible corrective rotation
    # --------------------------------------------------------
    for angle in corrective_angles:

        corrected = [
            rotate(img, angle)
            if angle != 0
            else img
            for img in probe_images
        ]

        # ====================================================
        # Branch 1: RAW LBP
        # ====================================================

        P_raw = desc.transform(
            corrected
        )

        D_raw = pairwise_distances(
            P_raw,
            G,
            "chi2"
        )

        # ====================================================
        # Branch 2: illumination-normalized LBP
        # ====================================================

        corrected_norm = [
            illumination_normalize(
                img,
                sigma=sigma
            )
            for img in corrected
        ]

        P_norm = desc.transform(
            corrected_norm
        )

        D_norm = pairwise_distances(
            P_norm,
            G_norm,
            "chi2"
        )

        # ====================================================
        # Fuse raw + normalized distances
        # ====================================================

        D_fused = (
            alpha * D_raw
            +
            (1 - alpha) * D_norm
        )

        all_fused_distances.append(
            D_fused
        )

    # --------------------------------------------------------
    # angles × probes × gallery
    # --------------------------------------------------------

    all_fused_distances = np.stack(
        all_fused_distances,
        axis=0
    )

    # --------------------------------------------------------
    # Best corrective angle for each probe-gallery pair
    # --------------------------------------------------------

    best_distances = np.min(
        all_fused_distances,
        axis=0
    )

    return recognition_rate(
        best_distances,
        gl,
        pl
    )

In [ ]:
final_normal = score_final_robust(
    prb
)

print("Baseline Normal:", 0.975)
print("Final Robust Normal:", final_normal)

Baseline Normal: 0.975
Final Robust Normal: 0.975


In [ ]:
# ============================================================
# Final evaluation:
# baseline vs final robust method
# ============================================================

final_results = []


def add_result(
    condition,
    probe_set
):

    baseline_acc = score(
        probe_set
    )

    final_acc = score_final_robust(
        probe_set
    )

    final_results.append({
        "Condition": condition,
        "Baseline": baseline_acc,
        "Final Robust": final_acc,
        "Gain": final_acc - baseline_acc
    })


# ------------------------------------------------------------
# 1. Normal
# ------------------------------------------------------------

add_result(
    "Normal",
    prb
)


# ------------------------------------------------------------
# 2. Uniform brightness
# ------------------------------------------------------------

uniform_probe = [
    0.5 * img + 20
    for img in prb
]

add_result(
    "Uniform brightness",
    uniform_probe
)


# ------------------------------------------------------------
# 3. Lighting ramps
# ------------------------------------------------------------

for s in [
    0.2,
    0.4,
    0.6,
    0.8
]:

    corrupted = [
        illum_ramp(
            img,
            s
        )
        for img in prb
    ]

    add_result(
        f"Lighting ramp {s}",
        corrupted
    )


# ------------------------------------------------------------
# 4. Rotations
# ------------------------------------------------------------

for d in [
    5,
    10,
    20
]:

    corrupted = [
        rotate(
            img,
            d
        )
        for img in prb
    ]

    add_result(
        f"Rotation {d}°",
        corrupted
    )


# ------------------------------------------------------------
# Final table
# ------------------------------------------------------------

final_df = pd.DataFrame(
    final_results
)

final_df

,Condition,Baseline,Final Robust,Gain
0,Normal,0.975,0.975,0.000
1,Uniform brightness,0.975,0.975,0.000
2,Lighting ramp 0.2,0.960,0.975,0.015
3,Lighting ramp 0.4,0.945,0.965,0.020
4,Lighting ramp 0.6,0.895,0.975,0.080
5,Lighting ramp 0.8,0.705,0.965,0.260
6,Rotation 5°,0.970,0.965,-0.005
7,Rotation 10°,0.925,0.955,0.030
8,Rotation 20°,0.630,0.880,0.250
